In [1]:
print("=" * 80)
print("Notebook 03 - Dataset Preparation")
print("=" * 80)
print("Starting dataset preparation pipeline...")

Notebook 03 - Dataset Preparation
Starting dataset preparation pipeline...


In [2]:
import os
from pathlib import Path

# Safe dynamic path resolution
try:
    from google.colab import drive
    if Path('/content').exists():
        drive.mount('/content/drive')
        PROJECT_DIR = Path('/content/drive/MyDrive/AI-Cybersecurity-Assistant')
    else:
        PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
except Exception:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

FINAL_PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "CIC-IDS2017_final"
)

print("PROJECT ROOT DIRECTORY:")
print(PROJECT_DIR)
print("
FINAL PROCESSED DATA DIRECTORY:")
print(FINAL_PROCESSED_DIR)


PROJECT AND DATASET DIRECTORIES
Project directory:
/content/drive/MyDrive/AI-Cybersecurity-Assistant

Final processed dataset directory:
/content/drive/MyDrive/AI-Cybersecurity-Assistant/data/processed/CIC-IDS2017_final

Directory exists: True


In [3]:
import glob

final_csv_files = sorted(
    glob.glob(
        os.path.join(
            FINAL_PROCESSED_DIR,
            "final_*.csv"
        )
    )
)

print("FINAL CLEANED CSV FILES")
print("=" * 80)

print("Files found:", len(final_csv_files))

for i, file in enumerate(final_csv_files, start=1):
    print(f"{i}. {os.path.basename(file)}")

FINAL CLEANED CSV FILES
Files found: 8
1. final_Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2. final_Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
3. final_Friday-WorkingHours-Morning.pcap_ISCX.csv
4. final_Monday-WorkingHours.pcap_ISCX.csv
5. final_Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
6. final_Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
7. final_Tuesday-WorkingHours.pcap_ISCX.csv
8. final_Wednesday-workingHours.pcap_ISCX.csv


In [4]:
import pandas as pd

print("FINAL DATASET COLUMN CHECK")
print("=" * 80)

reference_columns = None
all_columns_identical = True

for file in final_csv_files:
    columns = pd.read_csv(
        file,
        nrows=0
    ).columns.tolist()

    print(
        f"{os.path.basename(file)}: "
        f"{len(columns)} columns"
    )

    if reference_columns is None:
        reference_columns = columns
    elif columns != reference_columns:
        all_columns_identical = False

print("\n" + "=" * 80)

if all_columns_identical:
    print("✅ All 8 final CSV files have identical column names.")
else:
    print("⚠️ Column names differ between files.")

print("\nTotal columns:", len(reference_columns))
print("Target column:", reference_columns[-1])

FINAL DATASET COLUMN CHECK
final_Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 79 columns
final_Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 79 columns
final_Friday-WorkingHours-Morning.pcap_ISCX.csv: 79 columns
final_Monday-WorkingHours.pcap_ISCX.csv: 79 columns
final_Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79 columns
final_Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 79 columns
final_Tuesday-WorkingHours.pcap_ISCX.csv: 79 columns
final_Wednesday-workingHours.pcap_ISCX.csv: 79 columns

✅ All 8 final CSV files have identical column names.

Total columns: 79
Target column: Label


In [5]:
TARGET_COLUMN = "Label"

FEATURE_COLUMNS = [
    column
    for column in reference_columns
    if column != TARGET_COLUMN
]

print("FEATURE AND TARGET DEFINITION")
print("=" * 80)

print("Target column:", TARGET_COLUMN)
print("Number of features:", len(FEATURE_COLUMNS))

print("\nFirst 10 features:")
for i, feature in enumerate(
    FEATURE_COLUMNS[:10],
    start=1
):
    print(f"{i}. {feature}")

print("\nTarget included in features:",
      TARGET_COLUMN in FEATURE_COLUMNS)

FEATURE AND TARGET DEFINITION
Target column: Label
Number of features: 78

First 10 features:
1. Destination Port
2. Flow Duration
3. Total Fwd Packets
4. Total Backward Packets
5. Total Length of Fwd Packets
6. Total Length of Bwd Packets
7. Fwd Packet Length Max
8. Fwd Packet Length Min
9. Fwd Packet Length Mean
10. Fwd Packet Length Std

Target included in features: False


In [6]:
class_counts = {}

for file in final_csv_files:
    for chunk in pd.read_csv(
        file,
        usecols=["Label"],
        chunksize=100_000,
        low_memory=False
    ):
        counts = chunk["Label"].value_counts()

        for label, count in counts.items():
            class_counts[label] = (
                class_counts.get(label, 0) + int(count)
            )

class_labels = sorted(class_counts.keys())

label_to_id = {
    label: idx
    for idx, label in enumerate(class_labels)
}

id_to_label = {
    idx: label
    for label, idx in label_to_id.items()
}

print("LABEL ENCODING MAPPING")
print("=" * 80)

for idx in range(len(class_labels)):
    print(f"{idx}: {id_to_label[idx]}")

print("\nTotal classes:", len(class_labels))


LABEL ENCODING MAPPING
0: BENIGN
1: Bot
2: DDoS
3: DoS GoldenEye
4: DoS Hulk
5: DoS Slowhttptest
6: DoS slowloris
7: FTP-Patator
8: Infiltration
9: PortScan
10: SSH-Patator
11: Web Attack ' Brute Force
12: Web Attack ' Sql Injection
13: Web Attack ' XSS

Total classes: 14


In [7]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

print("TRAIN / TEST SPLIT CONFIGURATION")
print("=" * 80)

print("Training proportion:", f"{(1 - TEST_SIZE) * 100:.0f}%")
print("Testing proportion :", f"{TEST_SIZE * 100:.0f}%")
print("Random state       :", RANDOM_STATE)
print("Split strategy     : Stratified")

TRAIN / TEST SPLIT CONFIGURATION
Training proportion: 80%
Testing proportion : 20%
Random state       : 42
Split strategy     : Stratified


In [8]:
SPLIT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "CIC-IDS2017_splits"
)

os.makedirs(
    SPLIT_DIR,
    exist_ok=True
)

print("TRAIN / TEST OUTPUT DIRECTORY")
print("=" * 80)

print("Split directory:")
print(SPLIT_DIR)

print("\nDirectory exists:",
      os.path.exists(SPLIT_DIR))


TRAIN / TEST OUTPUT DIRECTORY
Split directory:
/content/drive/MyDrive/AI-Cybersecurity-Assistant/data/processed/CIC-IDS2017_splits

Directory exists: True


In [9]:
print("=" * 80)
print("        LOADING FINAL DATASET FOR TRAIN / TEST SPLIT")
print("=" * 80)

dataset_parts = []

for file in final_csv_files:
    print("Loading:", os.path.basename(file))

    df_part = pd.read_csv(
        file,
        low_memory=False
    )

    dataset_parts.append(df_part)

full_dataset = pd.concat(
    dataset_parts,
    ignore_index=True
)

print("\n" + "=" * 80)
print("FULL DATASET LOADED")
print("=" * 80)

print("Rows:", f"{len(full_dataset):,}")
print("Columns:", len(full_dataset.columns))

print("\nExpected rows: 2,394,375")

if len(full_dataset) == 2_394_375:
    print("✅ Row count matches final preprocessing result.")
else:
    print("⚠️ Row count does not match.")

        LOADING FINAL DATASET FOR TRAIN / TEST SPLIT
Loading: final_Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: final_Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: final_Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: final_Monday-WorkingHours.pcap_ISCX.csv
Loading: final_Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: final_Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: final_Tuesday-WorkingHours.pcap_ISCX.csv
Loading: final_Wednesday-workingHours.pcap_ISCX.csv

FULL DATASET LOADED
Rows: 2,394,375
Columns: 79

Expected rows: 2,394,375
✅ Row count matches final preprocessing result.


In [10]:
full_dataset["Label_ID"] = full_dataset["Label"].map(
    label_to_id
)

print("TARGET LABEL ENCODING")
print("=" * 80)

print("Original target column: Label")
print("Encoded target column: Label_ID")

print("\nMissing encoded labels:",
      int(full_dataset["Label_ID"].isna().sum()))

print("\nEncoded class distribution:")
print(
    full_dataset["Label_ID"]
    .value_counts()
    .sort_index()
)

print("\nNumber of encoded classes:",
      full_dataset["Label_ID"].nunique())

if full_dataset["Label_ID"].isna().sum() == 0:
    print("\n✅ All labels encoded successfully.")
else:
    print("\n⚠️ Some labels could not be encoded.")

TARGET LABEL ENCODING
Original target column: Label
Encoded target column: Label_ID

Missing encoded labels: 0

Encoded class distribution:
Label_ID
0     1977809
1        1948
2      128014
3        1122
4      172846
5        5228
6        5385
7        5931
8          36
9       90694
10       3219
11       1470
12         21
13        652
Name: count, dtype: int64

Number of encoded classes: 14

✅ All labels encoded successfully.


In [11]:
import numpy as np

X = full_dataset[FEATURE_COLUMNS].copy()
y = full_dataset["Label_ID"].copy()

print("FEATURE MATRIX AND TARGET VECTOR")
print("=" * 80)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX data type:")
print(type(X))

print("\ny data type:")
print(type(y))

print("\nNumber of features:", X.shape[1])
print("Number of target values:", len(y))

print("\nNon-numeric feature columns:")
print(
    X.select_dtypes(exclude="number").columns.tolist()
)

print("\nMissing feature values:",
      int(X.isna().sum().sum()))

print("\nInfinite feature values:",
      int(np.isinf(X).sum().sum()))

FEATURE MATRIX AND TARGET VECTOR
X shape: (2394375, 78)
y shape: (2394375,)

X data type:
<class 'pandas.core.frame.DataFrame'>

y data type:
<class 'pandas.core.series.Series'>

Number of features: 78
Number of target values: 2394375

Non-numeric feature columns:
[]

Missing feature values: 0

Infinite feature values: 0


In [12]:
print("CLASS COUNTS BEFORE TRAIN / TEST SPLIT")
print("=" * 80)

class_counts = y.value_counts().sort_index()

class_count_table = pd.DataFrame({
    "Class_ID": class_counts.index,
    "Label": [
        id_to_label[class_id]
        for class_id in class_counts.index
    ],
    "Count": class_counts.values
})

class_count_table["Percentage"] = (
    class_count_table["Count"]
    / len(y)
    * 100
)

display(class_count_table)

print("\n" + "=" * 80)
print("Total samples:", f"{len(y):,}")
print("Number of classes:", len(class_count_table))
print(
    "Smallest class:",
    int(class_count_table["Count"].min())
)

CLASS COUNTS BEFORE TRAIN / TEST SPLIT


,Class_ID,Label,Count,Percentage
0,0,BENIGN,1977809,82.602307
1,1,Bot,1948,0.081357
2,2,DDoS,128014,5.346447
3,3,DoS GoldenEye,1122,0.046860
4,4,DoS Hulk,172846,7.218836
5,5,DoS Slowhttptest,5228,0.218345
6,6,DoS slowloris,5385,0.224902
7,7,FTP-Patator,5931,0.247706
8,8,Infiltration,36,0.001504
9,9,PortScan,90694,3.787794



Total samples: 2,394,375
Number of classes: 14
Smallest class: 21


In [13]:
print("STRATIFIED SPLIT FEASIBILITY CHECK")
print("=" * 80)

split_check = class_count_table.copy()

split_check["Expected_Train"] = (
    split_check["Count"] * (1 - TEST_SIZE)
).round().astype(int)

split_check["Expected_Test"] = (
    split_check["Count"] * TEST_SIZE
).round().astype(int)

display(split_check)

print("\n" + "=" * 80)

print(
    "Minimum expected training samples:",
    split_check["Expected_Train"].min()
)

print(
    "Minimum expected testing samples:",
    split_check["Expected_Test"].min()
)

print("\nClasses with fewer than 10 expected test samples:")
display(
    split_check[
        split_check["Expected_Test"] < 10
    ][
        ["Class_ID", "Label", "Count", "Expected_Test"]
    ]
)

STRATIFIED SPLIT FEASIBILITY CHECK


,Class_ID,Label,Count,Percentage,Expected_Train,Expected_Test
0,0,BENIGN,1977809,82.602307,1582247,395562
1,1,Bot,1948,0.081357,1558,390
2,2,DDoS,128014,5.346447,102411,25603
3,3,DoS GoldenEye,1122,0.046860,898,224
4,4,DoS Hulk,172846,7.218836,138277,34569
5,5,DoS Slowhttptest,5228,0.218345,4182,1046
6,6,DoS slowloris,5385,0.224902,4308,1077
7,7,FTP-Patator,5931,0.247706,4745,1186
8,8,Infiltration,36,0.001504,29,7
9,9,PortScan,90694,3.787794,72555,18139



Minimum expected training samples: 17
Minimum expected testing samples: 4

Classes with fewer than 10 expected test samples:


,Class_ID,Label,Count,Expected_Test
8,8,Infiltration,36,7
12,12,Web Attack ' Sql Injection,21,4


In [14]:
from sklearn.model_selection import train_test_split

print("CREATING STRATIFIED TRAIN / TEST SPLIT")
print("=" * 80)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train features:", X_train.shape)
print("Test features :", X_test.shape)

print("Train target  :", y_train.shape)
print("Test target   :", y_test.shape)

print("\nTrain rows:",
      f"{len(X_train):,}")

print("Test rows :",
      f"{len(X_test):,}")

print("\n" + "=" * 80)

print("Expected train rows:",
      f"{int(len(X) * 0.80):,}")

print("Expected test rows:",
      f"{int(len(X) * 0.20):,}")

CREATING STRATIFIED TRAIN / TEST SPLIT
Train features: (1915500, 78)
Test features : (478875, 78)
Train target  : (1915500,)
Test target   : (478875,)

Train rows: 1,915,500
Test rows : 478,875

Expected train rows: 1,915,500
Expected test rows: 478,875


In [15]:
print("TRAIN / TEST CLASS DISTRIBUTION")
print("=" * 80)

train_counts = y_train.value_counts().sort_index()
test_counts = y_test.value_counts().sort_index()

split_distribution = pd.DataFrame({
    "Class_ID": range(len(class_labels)),
    "Label": [
        id_to_label[i]
        for i in range(len(class_labels))
    ],
    "Train_Count": [
        train_counts.get(i, 0)
        for i in range(len(class_labels))
    ],
    "Test_Count": [
        test_counts.get(i, 0)
        for i in range(len(class_labels))
    ]
})

split_distribution["Train_%"] = (
    split_distribution["Train_Count"]
    / len(y_train)
    * 100
)

split_distribution["Test_%"] = (
    split_distribution["Test_Count"]
    / len(y_test)
    * 100
)

display(split_distribution)

print("\n" + "=" * 80)

print("Classes in training set:",
      y_train.nunique())

print("Classes in testing set:",
      y_test.nunique())

if (
    y_train.nunique() == 14
    and y_test.nunique() == 14
):
    print("\n✅ All 14 classes are present in both sets.")
else:
    print("\n⚠️ One or more classes are missing.")

TRAIN / TEST CLASS DISTRIBUTION


,Class_ID,Label,Train_Count,Test_Count,Train_%,Test_%
0,0,BENIGN,1582247,395562,82.602297,82.602349
1,1,Bot,1558,390,0.081336,0.081441
2,2,DDoS,102411,25603,5.346437,5.346489
3,3,DoS GoldenEye,898,224,0.046881,0.046776
4,4,DoS Hulk,138277,34569,7.218846,7.218794
5,5,DoS Slowhttptest,4182,1046,0.218324,0.218429
6,6,DoS slowloris,4308,1077,0.224902,0.224902
7,7,FTP-Patator,4745,1186,0.247716,0.247664
8,8,Infiltration,29,7,0.001514,0.001462
9,9,PortScan,72555,18139,3.787784,3.787836



Classes in training set: 14
Classes in testing set: 14

✅ All 14 classes are present in both sets.


In [16]:
TRAIN_FILE = os.path.join(
    SPLIT_DIR,
    "train.csv"
)

TEST_FILE = os.path.join(
    SPLIT_DIR,
    "test.csv"
)

print("SAVING TRAIN / TEST SPLIT")
print("=" * 80)

print("Training file:")
print(TRAIN_FILE)

print("\nTesting file:")
print(TEST_FILE)

# Save training data
train_data = X_train.copy()
train_data["Label_ID"] = y_train.values

train_data.to_csv(
    TRAIN_FILE,
    index=False
)

# Save testing data
test_data = X_test.copy()
test_data["Label_ID"] = y_test.values

test_data.to_csv(
    TEST_FILE,
    index=False
)

print("\n" + "=" * 80)
print("✅ TRAIN / TEST SPLIT SAVED")
print("=" * 80)

print("Training rows:", f"{len(train_data):,}")
print("Testing rows :", f"{len(test_data):,}")

SAVING TRAIN / TEST SPLIT
Training file:
/content/drive/MyDrive/AI-Cybersecurity-Assistant/data/processed/CIC-IDS2017_splits/train.csv

Testing file:
/content/drive/MyDrive/AI-Cybersecurity-Assistant/data/processed/CIC-IDS2017_splits/test.csv

✅ TRAIN / TEST SPLIT SAVED
Training rows: 1,915,500
Testing rows : 478,875


In [17]:
print("SAVED TRAIN / TEST FILE VERIFICATION")
print("=" * 80)

print("Training file exists:", os.path.exists(TRAIN_FILE))
print("Testing file exists :", os.path.exists(TEST_FILE))

train_saved = pd.read_csv(
    TRAIN_FILE,
    nrows=5
)

test_saved = pd.read_csv(
    TEST_FILE,
    nrows=5
)

print("\nTraining file columns:", len(train_saved.columns))
print("Testing file columns :", len(test_saved.columns))

print("\nExpected columns:", len(FEATURE_COLUMNS) + 1)

print("\nTraining file size:")
print(f"{os.path.getsize(TRAIN_FILE) / (1024**3):.2f} GB")

print("Testing file size:")
print(f"{os.path.getsize(TEST_FILE) / (1024**3):.2f} GB")

if (
    os.path.exists(TRAIN_FILE)
    and os.path.exists(TEST_FILE)
    and len(train_saved.columns) == 79
    and len(test_saved.columns) == 79
):
    print("\n✅ TRAIN / TEST FILES VERIFIED")
else:
    print("\n⚠️ FILE VERIFICATION REQUIRES ATTENTION")

SAVED TRAIN / TEST FILE VERIFICATION
Training file exists: True
Testing file exists : True

Training file columns: 79
Testing file columns : 79

Expected columns: 79

Training file size:
0.62 GB
Testing file size:
0.15 GB

✅ TRAIN / TEST FILES VERIFIED


In [18]:
import json

split_metadata = {
    "dataset": "CIC-IDS2017",
    "total_rows": 2_394_375,
    "training_rows": 1_915_500,
    "testing_rows": 478_875,
    "features": 78,
    "classes": 14,
    "target_column": "Label_ID",
    "original_label_column": "Label",
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "split_strategy": "Stratified",
    "label_mapping": {
        str(label_id): label
        for label_id, label in id_to_label.items()
    }
}

split_metadata_path = os.path.join(
    SPLIT_DIR,
    "split_metadata.json"
)

with open(split_metadata_path, "w") as f:
    json.dump(
        split_metadata,
        f,
        indent=4
    )

print("=" * 80)
print("        TRAIN / TEST SPLIT METADATA SAVED")
print("=" * 80)

print("Metadata file:")
print(split_metadata_path)

print("\nTraining rows:",
      split_metadata["training_rows"])

print("Testing rows:",
      split_metadata["testing_rows"])

print("Classes:",
      split_metadata["classes"])

print("\n✅ Split metadata saved successfully.")

        TRAIN / TEST SPLIT METADATA SAVED
Metadata file:
/content/drive/MyDrive/AI-Cybersecurity-Assistant/data/processed/CIC-IDS2017_splits/split_metadata.json

Training rows: 1915500
Testing rows: 478875
Classes: 14

✅ Split metadata saved successfully.


In [1]:
# Notebook 03 - Cell 19
# Memory-safe final validation

import os
import glob
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/AI-Cybersecurity-Assistant"

SPLIT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "CIC-IDS2017_splits"
)

TRAIN_FILE = os.path.join(
    SPLIT_DIR,
    "train.csv"
)

TEST_FILE = os.path.join(
    SPLIT_DIR,
    "test.csv"
)

CHUNK_SIZE = 100_000

print("=" * 80)
print("        NOTEBOOK 03 — MEMORY-SAFE FINAL VALIDATION")
print("=" * 80)

print("\nTraining file exists:", os.path.exists(TRAIN_FILE))
print("Testing file exists :", os.path.exists(TEST_FILE))

def validate_split_file(file_path, expected_rows):

    total_rows = 0
    total_missing = 0
    total_infinite = 0
    labels_found = set()
    column_count = None

    for chunk in pd.read_csv(
        file_path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        total_rows += len(chunk)

        if column_count is None:
            column_count = len(chunk.columns)

        feature_cols = [
            column
            for column in chunk.columns
            if column != "Label_ID"
        ]

        total_missing += int(
            chunk[feature_cols]
            .isna()
            .sum()
            .sum()
        )

        total_infinite += int(
            np.isinf(chunk[feature_cols]).sum()
            .sum()
        )

        labels_found.update(
            chunk["Label_ID"]
            .dropna()
            .unique()
        )

    return {
        "rows": total_rows,
        "columns": column_count,
        "features": len(feature_cols),
        "missing": total_missing,
        "infinite": total_infinite,
        "classes": len(labels_found),
        "expected_rows": expected_rows
    }


print("\nValidating training file...")
train_result = validate_split_file(
    TRAIN_FILE,
    1_915_500
)

print("Training validation complete.")

print("\nValidating testing file...")
test_result = validate_split_file(
    TEST_FILE,
    478_875
)

print("Testing validation complete.")

print("\n" + "=" * 80)
print("TRAINING SET")
print("-" * 80)

print("Rows:", f"{train_result['rows']:,}")
print("Columns:", train_result["columns"])
print("Features:", train_result["features"])
print("Classes:", train_result["classes"])
print("Missing values:", train_result["missing"])
print("Infinite values:", train_result["infinite"])

print("\n" + "=" * 80)
print("TESTING SET")
print("-" * 80)

print("Rows:", f"{test_result['rows']:,}")
print("Columns:", test_result["columns"])
print("Features:", test_result["features"])
print("Classes:", test_result["classes"])
print("Missing values:", test_result["missing"])
print("Infinite values:", test_result["infinite"])

print("\n" + "=" * 80)

if (
    train_result["rows"] == 1_915_500
    and test_result["rows"] == 478_875
    and train_result["columns"] == 79
    and test_result["columns"] == 79
    and train_result["features"] == 78
    and test_result["features"] == 78
    and train_result["classes"] == 14
    and test_result["classes"] == 14
    and train_result["missing"] == 0
    and test_result["missing"] == 0
    and train_result["infinite"] == 0
    and test_result["infinite"] == 0
):
    print("✅ NOTEBOOK 03 DATASET SPLIT PASSED FINAL VALIDATION")
else:
    print("⚠️ DATASET SPLIT REQUIRES FURTHER VALIDATION")

        NOTEBOOK 03 — MEMORY-SAFE FINAL VALIDATION

Training file exists: True
Testing file exists : True

Validating training file...
Training validation complete.

Validating testing file...
Testing validation complete.

TRAINING SET
--------------------------------------------------------------------------------
Rows: 1,915,500
Columns: 79
Features: 78
Classes: 14
Missing values: 0
Infinite values: 0

TESTING SET
--------------------------------------------------------------------------------
Rows: 478,875
Columns: 79
Features: 78
Classes: 14
Missing values: 0
Infinite values: 0

✅ NOTEBOOK 03 DATASET SPLIT PASSED FINAL VALIDATION


In [3]:
# Notebook 03 - Cell 20
# Final Dataset Preparation Report
# RAM-safe and independent of previous notebook variables

print("=" * 90)
print("        NOTEBOOK 03 — FINAL DATASET PREPARATION REPORT")
print("=" * 90)

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------
print("\nDATASET")
print("-" * 90)

print("Dataset: CIC-IDS2017")
print("Final dataset rows:", f"{2_394_375:,}")
print("Final dataset columns:", 79)
print("Feature columns:", 78)
print("Target column: Label_ID")
print("Number of classes:", 14)

# ---------------------------------------------------------
# LABEL ENCODING
# ---------------------------------------------------------
label_mapping = {
    0: "BENIGN",
    1: "Bot",
    2: "DDoS",
    3: "DoS GoldenEye",
    4: "DoS Hulk",
    5: "DoS Slowhttptest",
    6: "DoS slowloris",
    7: "FTP-Patator",
    8: "Infiltration",
    9: "PortScan",
    10: "SSH-Patator",
    11: "Web Attack ' Brute Force",
    12: "Web Attack ' Sql Injection",
    13: "Web Attack ' XSS"
}

print("\nLABEL ENCODING")
print("-" * 90)

for class_id, label in label_mapping.items():
    print(f"{class_id:2d} -> {label}")

# ---------------------------------------------------------
# TRAIN / TEST SPLIT
# ---------------------------------------------------------
print("\nTRAIN / TEST SPLIT")
print("-" * 90)

print("Split strategy: Stratified")
print("Training proportion: 80%")
print("Testing proportion: 20%")
print("Random state: 42")

# ---------------------------------------------------------
# TRAINING SET
# ---------------------------------------------------------
print("\nTRAINING SET")
print("-" * 90)

print("Rows:", f"{1_915_500:,}")
print("Features:", 78)
print("Classes:", 14)
print("Missing values:", 0)
print("Infinite values:", 0)

# ---------------------------------------------------------
# TESTING SET
# ---------------------------------------------------------
print("\nTESTING SET")
print("-" * 90)

print("Rows:", f"{478_875:,}")
print("Features:", 78)
print("Classes:", 14)
print("Missing values:", 0)
print("Infinite values:", 0)

# ---------------------------------------------------------
# OUTPUT FILES
# ---------------------------------------------------------
print("\nOUTPUT FILES")
print("-" * 90)

print("Training file:")
print(
    "/content/drive/MyDrive/AI-Cybersecurity-Assistant/"
    "data/processed/CIC-IDS2017_splits/train.csv"
)

print("\nTesting file:")
print(
    "/content/drive/MyDrive/AI-Cybersecurity-Assistant/"
    "data/processed/CIC-IDS2017_splits/test.csv"
)

print("\nSplit metadata:")
print(
    "/content/drive/MyDrive/AI-Cybersecurity-Assistant/"
    "data/processed/CIC-IDS2017_splits/split_metadata.json"
)

# ---------------------------------------------------------
# FINAL STATUS
# ---------------------------------------------------------
print("\nDATASET QUALITY")
print("-" * 90)

print("All 14 classes present in training: YES")
print("All 14 classes present in testing: YES")
print("Train/test class distribution preserved: YES")
print("Saved files verified: YES")
print("Memory-safe final validation: PASSED")

print("\n" + "=" * 90)
print("✅ NOTEBOOK 03 — DATASET PREPARATION COMPLETE")
print("=" * 90)
print("Ready for Notebook 04 — Baseline Machine Learning Models")
print("=" * 90)

        NOTEBOOK 03 — FINAL DATASET PREPARATION REPORT

DATASET
------------------------------------------------------------------------------------------
Dataset: CIC-IDS2017
Final dataset rows: 2,394,375
Final dataset columns: 79
Feature columns: 78
Target column: Label_ID
Number of classes: 14

LABEL ENCODING
------------------------------------------------------------------------------------------
 0 -> BENIGN
 1 -> Bot
 2 -> DDoS
 3 -> DoS GoldenEye
 4 -> DoS Hulk
 5 -> DoS Slowhttptest
 6 -> DoS slowloris
 7 -> FTP-Patator
 8 -> Infiltration
 9 -> PortScan
10 -> SSH-Patator
11 -> Web Attack ' Brute Force
12 -> Web Attack ' Sql Injection
13 -> Web Attack ' XSS

TRAIN / TEST SPLIT
------------------------------------------------------------------------------------------
Split strategy: Stratified
Training proportion: 80%
Testing proportion: 20%
Random state: 42

TRAINING SET
------------------------------------------------------------------------------------------
Rows: 1,915,500
F